# 2. Google News Local pages, Ontario municipalities

Reads each town's Google News Local page (logged out, English and French) and saves its story cards. A page can't be read for a past date, so each run captures that day only.

- **Input:** `data/raw/local_page_urls.csv`, one Local page address per town. Each address is built from the town's Google Knowledge Graph ID (from Wikidata, checked against its StatCan code).
- **Output:** `outputs/local_page_cards_<run>.csv`.

At 30 s between requests, the 40-town sample (`SAMPLE_ONLY = True`, 80 pages) takes about 45 minutes; all 390 towns (780 pages) take about 7 hours. The run stops at a block or a CAPTCHA.

In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'collection' else Path.cwd()   # repository root
RAW, CACHE, OUT = ROOT / 'data' / 'raw', ROOT / 'cache', ROOT / 'outputs'
CACHE.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
import datetime as dt, gzip, html, random, re, time
import pandas as pd, requests

SAMPLE_ONLY = True     # True = the 40-town sample; False = every town
GAP = 30               # seconds between requests
EDITIONS = {'en-CA': 'hl=en-CA&gl=CA&ceid=CA:en', 'fr-CA': 'hl=fr-CA&gl=CA&ceid=CA:fr'}
STOP = ('consent.google.com', '/sorry/', 'unusual traffic', 'captcha')
HEADERS = {'User-Agent': 'INSE6450 student research (Google News Local pages)'}

towns = pd.read_csv(RAW / 'local_page_urls.csv', dtype={'town_id': str})
if SAMPLE_ONLY:
    towns = towns[towns.sample_40]
print(f"{len(towns)} towns x {len(EDITIONS)} editions = {len(towns) * len(EDITIONS)} pages")

40 towns x 2 editions = 80 pages


## Step 1. Read the pages

In [2]:
run = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%MZ')
run_dir = CACHE / 'local_pages' / run
run_dir.mkdir(parents=True, exist_ok=True)
for i, (t, ed) in enumerate((t, ed) for t in towns.itertuples() for ed in EDITIONS):
    if i:
        time.sleep(GAP + random.uniform(0, GAP / 4))
    r = requests.get(f'{t.topic_url}?{EDITIONS[ed]}', headers={**HEADERS, 'Accept-Language': ed}, timeout=60)
    print(f'{i + 1}. {t.short_name} {ed}: {r.status_code}')
    if r.status_code in (403, 429, 503) or any(s in r.url or s in r.text[:20000].lower() for s in STOP):
        print('Stopped: Google answered with a block or a CAPTCHA. Try again another day.')
        break
    if r.status_code == 200:
        with gzip.open(run_dir / f'{t.town_id}_{ed}.html.gz', 'wt', encoding='utf-8') as f:
            f.write(r.text)

1. Armstrong en-CA: 200


KeyboardInterrupt: 

## Step 2. Read the story cards

Each card on the page has:
- a hidden link (`class="WwrzSb"`) whose `jslog` attribute carries the original article URL in base64;
- the outlet's name in `class="vr1PYe"`;
- the time in `<time datetime>`;
- the headline in a `./read/...` link.

Cards are numbered in page order (`rank`).

In [ ]:
import base64

def b64text(s):
    try:
        return base64.b64decode(s + '=' * (-len(s) % 4)).decode('utf-8', 'replace')
    except ValueError:
        return ''

def parse_cards(page):
    starts = [m.start() for m in re.finditer(r'<a class="WwrzSb"', page)]
    cards, seen = [], set()
    for k, st in enumerate(starts):
        chunk = page[st: starts[k + 1] if k + 1 < len(starts) else st + 20000]
        link = re.search(r'jslog="95014; 5:([A-Za-z0-9+/=]+)', chunk)
        url = re.search(r'https?://[^"\\]+', b64text(link.group(1))) if link else None
        source = re.search(r'class="vr1PYe"[^>]*>([^<]+)<', chunk)
        when = re.search(r'datetime="([^"]+)"', chunk)
        heads = [html.unescape(re.sub(r'<[^>]+>', '', h)).strip()
                 for h in re.findall(r'<a[^>]*href="\./read/[^"]*"[^>]*>(.*?)</a>', chunk, flags=re.S)]
        head = max([h for h in heads if h], key=len, default='')
        if head and head not in seen:
            seen.add(head)
            cards.append({'rank': len(cards) + 1, 'headline': head, 'source': html.unescape(source.group(1)) if source else '',
                          'time': when.group(1) if when else '', 'link': url.group(0) if url else ''})
    return cards

rows = []
for p in sorted(run_dir.glob('*.html.gz')):
    town_id, ed = p.name.split('.')[0].split('_')
    page = gzip.open(p, 'rt', encoding='utf-8', errors='replace').read()
    rows += [{'run': run, 'town_id': town_id, 'edition': ed, **c} for c in parse_cards(page)]
cards = pd.DataFrame(rows)
cards.to_csv(OUT / f'local_page_cards_{run}.csv', index=False)
print(f'{cards.town_id.nunique()} towns, {len(cards)} cards')
cards.head()

## Step 3. Quick look: how much of each page is about the town?

A rough check: the share of cards whose headline names the town. The tags in `data/google_news_local_pages/` are stricter: they also count the town's communities and its own outlets, and every race story was checked by hand.

In [ ]:
c = cards.merge(towns[['town_id', 'short_name', 'population_2021']], on='town_id')
c['names_town'] = [n.lower() in h.lower() for n, h in zip(c.short_name, c.headline)]
by_town = c[c.edition == 'en-CA'].groupby(['short_name', 'population_2021']).names_town.mean().reset_index()
by_town['size'] = pd.cut(by_town.population_2021, [0, 2000, 10000, 50000, 1e8], labels=['<2k', '2k-10k', '10k-50k', '50k+'])
by_town.groupby('size', observed=True).names_town.agg(['mean', 'count']).rename(columns={'mean': 'share of cards naming the town', 'count': 'towns'})

## Step 4. Data report: missing values, duplicates, example rows

Reads the finished tables in `data/` (it does not use the pages read above).
Saves `outputs/local_missingness.csv` (every column with missing values) and prints the numbers used in Section 2 of the report.

In [ ]:
from pathlib import Path
import pandas as pd

# Setting root
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'google_news_local_pages').is_dir())
LP = ROOT / 'data' / 'google_news_local_pages'
TW = ROOT / 'data' / 'ontario_towns'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)

tables = {'local_page_cards': pd.read_csv(LP / 'local_page_cards.csv', low_memory=False),
          'repeat_readings': pd.read_csv(LP / 'local_page_cards_repeat_readings.csv'),
          'local_page_by_town': pd.read_csv(LP / 'local_page_by_town.csv'),
          'towns': pd.read_csv(TW / 'towns.csv'),
          'races_2026': pd.read_csv(TW / 'races_2026.csv'),
          'local_outlets': pd.read_csv(TW / 'local_outlets.csv', low_memory=False),
          'local_stories': pd.read_csv(TW / 'local_stories.csv', low_memory=False),
          'civic_outcomes': pd.read_csv(TW / 'civic_outcomes.csv')}

# new names, so the cards read in Step 2 are not overwritten
all_cards, rep, bytown, stories, civ = (tables[k] for k in
    ['local_page_cards', 'repeat_readings', 'local_page_by_town', 'local_stories', 'civic_outcomes'])

# missing values: every column with any, saved for the report
miss = pd.concat([pd.DataFrame({'table': n, 'column': d.columns, 'n_missing': d.isna().sum().values,
                                'pct_missing': (d.isna().mean() * 100).round(1).values}) for n, d in tables.items()])
miss = miss[miss.n_missing > 0]
miss.to_csv(OUT / 'local_missingness.csv', index=False)

read = bytown[bytown.outcome != 'not_collected'] # the towns that were read
days = set(pd.to_datetime(pd.concat([all_cards.run, rep.run]), format='%Y%m%dT%H%MZ').dt.date)

print('Towns not collected:', (bytown.outcome == 'not_collected').sum(), 'of', len(bytown))
print('Empty columns (to drop):', [c for c in bytown.columns if bytown[c].isna().all()])
print('Towns read with 0 French cards:', (read['cards_fr-CA'] == 0).sum(), 'of', len(read))
print('Turnout missing:', civ.turnout_2022.isna().sum(), '| all with a fully acclaimed council:',
      civ.loc[civ.turnout_2022.isna(), 'council_fully_acclaimed_2022'].all())
print('Days with no reading (Oct 12-16):', [str(d) for d in pd.date_range('2026-10-12', '2026-10-16').date if d not in days])
print('Election cards about the town:', int(all_cards.race_card.sum()), 'of', len(all_cards),
      '| towns read under 10,000:', (read.population_2021 < 10000).sum(), 'of', len(read))
print('Same link twice on a page:', all_cards.duplicated(['town_id', 'edition', 'link']).sum(),
      '| syndicated titles per town:', stories.duplicated(['title_norm', 'town_id']).sum(),
      '| most stories for one town:', stories.groupby('town_id').size().max())

# example rows (section 2.3): two cards about the town's own race, one about another town's race
cols = ['town_id', 'edition', 'rank', 'headline', 'source', 'time', 'race_card', 'other_town_race']
display(pd.concat([all_cards[all_cards.race_card].head(2), all_cards[all_cards.other_town_race.notna()].head(1)])[cols])
miss

Towns not collected: 4 of 394
Empty columns (to drop): ['stratum', 'probe_headline', 'probe_shown', 'web_recall_tiers']
Towns read with 0 French cards: 256 of 390
Turnout missing: 32 | all with a fully acclaimed council: True
Days with no reading (Oct 12-16): ['2026-10-14', '2026-10-15', '2026-10-16']
Election cards about the town: 360 of 21461 | towns read under 10,000: 239 of 390
Same link twice on a page: 13 | syndicated titles per town: 11533 | most stories for one town: 18397


,town_id,edition,rank,headline,source,time,race_card,other_town_race
349,3502044,fr-CA,16,Un conseiller de Casselman veut faire le saut ...,Le Droit,2026-01-18T08:00:00Z,True,NaN
573,3511005,en-CA,59,"Six questions with Paul Proderick, candidate f...",Kingstonist News,2026-09-24T07:00:00Z,True,NaN
1,3501005,en-CA,2,Homelessness the top issue at Cornwall mayoral...,Cornwall Standard Freeholder,2026-10-09T20:44:03Z,False,3501012


,table,column,n_missing,pct_missing
16,local_page_cards,other_town_race,20650,96.2
5,local_page_by_town,french_mt_share,1,0.3
6,local_page_by_town,franco_band,1,0.3
7,local_page_by_town,stratum,394,100.0
9,local_page_by_town,probe_headline,394,100.0
...,...,...,...,...
6,local_stories,title,585,1.0
7,local_stories,title_clean,585,1.0
8,local_stories,title_norm,1225,2.1
14,local_stories,home_veto,58088,98.9
